# Stage 2 Notebook 75 - Joint BDD training initialized from NB74's CULane-pretrained backbone

**Priorities 2 + 3 combined.** NB74 produced a CULane-pretrained `backbone_pretrained.pt`. This notebook loads it as init for a joint BDD100K run with the speed flags from NB73.

Why this matters: across 40+ experiments the backbone was always random-init. The reason mAP50 stuck near 0 at full data and matched_iou plateaued around 0.55 is partly that the backbone is starting from noise on EVERY run. A CULane-trained backbone should give:
- Faster joint convergence (det converges to mAP > 0 in 8-12 epochs instead of needing 30+)
- Higher lane ceiling (matched_iou > 0.60 instead of ~0.55)
- Possibly cracks the cls collapse (cls features start from a lane-aware basin)

### Run mode
1. Confirm NB74 produced `backbone_pretrained.pt` in Drive.
2. Smoke.
3. 8 epochs full BDD with the CULane init + speed flags + torch.compile. ~2 hr.

In [ ]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [ ]:
# Extract NB74's backbone-only checkpoint from its output tar.
from pathlib import Path
import os, sys, subprocess

NB74_TAR = '/content/drive/MyDrive/EcoCAR/training_runs/exp69_rmt_gca_culane_lane_only_pretrain_culane8.tar'
PRETRAIN_DIR = '/content/exp69_extracted'
PRETRAIN_BACKBONE = f'{PRETRAIN_DIR}/backbone_pretrained.pt'

Path(PRETRAIN_DIR).mkdir(parents=True, exist_ok=True)
print(f'Extracting CULane-pretrained backbone from {NB74_TAR}...')
subprocess.check_call(['tar', '-xf', NB74_TAR, '-C', PRETRAIN_DIR])
assert Path(PRETRAIN_BACKBONE).exists(), f'{PRETRAIN_BACKBONE} not in tar -- did NB74 finish?'
print(f'Pretrained backbone ready at {PRETRAIN_BACKBONE}')
subprocess.check_call(['ls', '-lh', PRETRAIN_BACKBONE])

Extracting CULane-pretrained backbone from /content/drive/MyDrive/EcoCAR/training_runs/exp69_rmt_gca_culane_lane_only_pretrain_culane8.tar...
Pretrained backbone ready at /content/exp69_extracted/backbone_pretrained.pt


0

In [ ]:
# Joint BDD training with CULane-pretrained backbone + speed flags.
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp70_rmt_gca_culane_init_anchor_cls_sep_vfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'
PRETRAIN_BACKBONE = '/content/exp69_extracted/backbone_pretrained.pt'

DEBUG_MODE = False
if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 8
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full8_culane_init'
    EPOCHS = 8
    BATCH_SIZE = 32
    LIMIT_TRAIN = None
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
    # PRETRAINED INIT:
    '--pretrained-backbone', PRETRAIN_BACKBONE,
    # SPEED FLAGS:
    '--workers', '6',
    '--prefetch-factor', '4',
    '--torch-compile',
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, 'BATCH_SIZE:', BATCH_SIZE, flush=True)
print('Using pretrained backbone:', PRETRAIN_BACKBONE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False BATCH_SIZE: 32
Using pretrained backbone: /content/exp69_extracted/backbone_pretrained.pt
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp70_rmt_gca_culane_init_anchor_cls_sep_vfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp70_rmt_gca_culane_init_anchor_cls_sep_vfl_joint_full8_culane_init --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp70_rmt_gca_culane_init_anchor_cls_sep_vfl_joint_full8_culane_init.tar --epochs 8 --batch-size 32 --limit-val 1000 --force-extract --print-every 50 --pretrained-backbone /content/exp69_extracted/backbone_pretrained.pt --workers 6 --prefetch-factor 4 --torch-compile
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp70_rmt_gca_culane_init_anchor_cls_sep_vfl_joint.yaml --curve-tar /content/drive/M

## What to watch in NB75 (CULane-init joint training)

Reference NB62 (random-init backbone, 12 ep full): matched_iou=0.55, decoded_f1=0.06, val_map50=0.
Reference NB45 (random-init backbone, 3K data, 30 ep, width=1.0): val_map50=0.011 -- our best det.

Pass criteria at epoch 8 (40% fewer epochs than NB62 but with pretrained init):
- **`val/matched_line_iou >= 0.60`** -- the CULane-trained backbone has lane-aware features.
- **`val/lane/decoded_f1 >= 0.10`** -- 1.7x NB62's 0.06; the lane head's cls problem partially fixed by better initial backbone features.
- **`val_map50 >= 0.01`** -- det actually starts converging because random-init handicap is gone.
- `[pretrained] loaded N/M tensors` log line shows reasonable load coverage (50%+).
- Wall-clock 1-2 hr (vs NB62's 14 hr).

If matched_iou >= 0.60 AND val_map50 >= 0.01: **the pretrained-init is the single biggest lift** we've had in 75 experiments. Stage 3 deployment uses this checkpoint.

If pretrained loaded < 30% of tensors: the loader's name-matching is too strict. Check the log for the first few source/target keys and adjust `_normalize_keys` in pretrained_loader.py.

If metrics don't improve over NB62: CULane pretrain was too short OR architecture mismatch. Retry with end_epoch=16 on NB74.